In [1]:
%pip install -U torch transformers datasets accelerate evaluate scikit-learn huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 17.3 MB/s  0:00:22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 23.1 MB/s  0:00:18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 26.5 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 26.1 MB/s  0:00:13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 37.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 33.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 32.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 21.3 MB/s  0:00:10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 MB 20.2 MB/s  0:00:11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.0/151.0 MB 27.6 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 MB 21.1 MB/s  0:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 25.7 MB/s  0:00:00


In [14]:
!pip uninstall -y torch torchaudio torchvision
!pip install torch torchvision torchaudio

Found existing installation: torch 2.15.0
Uninstalling torch-2.15.0:
  Successfully uninstalled torch-2.15.0
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Found existing installation: torchvision 0.29.1+cu130
Uninstalling torchvision-0.29.1+cu130:
  Successfully uninstalled torchvision-0.29.1+cu130
  Using cached torch-2.15.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (36 kB)
Using cached torch-2.15.0-cp313-cp313-manylinux_2_28_x86_64.whl (539.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 17.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchaudio]


In [3]:
from huggingface_hub import login

login()

In [4]:
import os
import numpy as np
import pandas as pd
import torch

#from datasets import Dataset, Datasetdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
CSV_path = "bengali_sentences_final.csv"

TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

print("CUDA available:", torch.cuda.is_available())
print("PyTorch device:", "cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
PyTorch device: cuda


In [5]:
df = pd.read_csv("bengali_sentences_final.csv")

# Select columns: sentence (1st col in selection), positive, negative, question, happiness(7th col), sadness(10th col)
selected_df = pd.concat([
    df.iloc[:, 0],   # sentence
    df.iloc[:, 1],   # positive
    df.iloc[:, 2],   # negative
    df.iloc[:, 3],   # question
    df.iloc[:, 4],   # happiness (7th col, index 6)
    df.iloc[:, 5]    # sadness (10th col, index 9)
], axis=1).fillna(0)

# Rename columns
selected_df.columns = ['sentence', 'Positive', 'Negative', 'Question', 'Happiness', 'Sadness']

print(selected_df.head())

                                            sentence  Positive  Negative  \
0                          পড়ছি আর নিজের মনে হাসছি।         1         0   
1        বাবা টেনি, তুমি আমায় ধনে-প্রাণে বাঁচিয়েছ।         1         0   
2  কাঙ্গা নেপালীর ছেলে, জান দিয়ে দেবে, কিন্তু মন...         0         1   
3                  এ যাত্রা আপনি ওকে নিয়ে যাবেন না।         0         1   
4                                       কিছু কি নেই?         0         1   

   Question  Happiness  Sadness  
0         0          1        0  
1         0          1        0  
2         0          0        0  
3         0          0        0  
4         1          0        0  


In [6]:
hf_dataset = Dataset.from_pandas(selected_df)

def process_labels(example):
    example["labels"] = [
        float(example["Positive"]),
        float(example["Happiness"]),
        float(example["Negative"]),
        float(example["Sadness"]),
        float(example["Question"])
    ]
    return example

hf_dataset = hf_dataset.map(process_labels)


# Remove original separate columns
hf_dataset = hf_dataset.remove_columns(['Positive', 'Happiness', 'Negative',  'Sadness', 'Question'])
hf_dataset
#print(hf_dataset[0])

Map:   0%|          | 0/5182 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence', 'labels'],
    num_rows: 5182
})

In [7]:
# Convert the Hugging Face dataset to a DataFrame
df1 = hf_dataset.to_pandas()

# Remove rows with no active labels
#df1 = df[df["labels"].apply(lambda x: sum(x) > 0)].reset_index(drop=True)


train_df, temp_df = train_test_split(
    df1,
    test_size = 0.2,
    random_state = 42,

)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,

)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

Training samples: 4145
Validation samples: 518
Test samples: 519


In [8]:
print("Train columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

print("\nFirst training example:")
print(train_df.iloc[0])

print("\nData types:")
print(train_df.dtypes)

Train columns: ['sentence', 'labels']
Validation columns: ['sentence', 'labels']
Test columns: ['sentence', 'labels']

First training example:
sentence    তাই প্রমোদের চিঠিটা পেয়ে মনটা খুশিতে ভরে উঠল।
labels                           [1.0, 1.0, 0.0, 0.0, 0.0]
Name: 1128, dtype: object

Data types:
sentence    object
labels      object
dtype: object


In [9]:
from datasets import Dataset, DatasetDict, Features
from datasets import Value, Sequence

# Define the dataset schema
features = Features({
    "sentence": Value("string"),
    "labels": Sequence(Value("float32"), length=5)
})

# Convert each split
hf_dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[["sentence", "labels"]],
        features=features,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        val_df[["sentence", "labels"]],
        features=features,
        preserve_index=False
    ),
    "test": Dataset.from_pandas(
        test_df[["sentence", "labels"]],
        features=features,
        preserve_index=False
    )
})

print(hf_dataset)
print("\nOne training example:")
print(hf_dataset["train"][0])

print("\nDataset features:")
print(hf_dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'labels'],
        num_rows: 4145
    })
    validation: Dataset({
        features: ['sentence', 'labels'],
        num_rows: 518
    })
    test: Dataset({
        features: ['sentence', 'labels'],
        num_rows: 519
    })
})

One training example:
{'sentence': 'তাই প্রমোদের চিঠিটা পেয়ে মনটা খুশিতে ভরে উঠল।', 'labels': [1.0, 1.0, 0.0, 0.0, 0.0]}

Dataset features:
{'sentence': Value('string'), 'labels': List(Value('float32'), length=5)}


In [10]:
# Pretrained model to download
BASE_MODEL = "google-bert/bert-base-multilingual-cased"

# New Hugging Face repository for the trained model
NEW_MODEL_REPO = "Subha95/bengali-sentiment-bert"

In [11]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Tokenizer loaded:", BASE_MODEL)
print(tokenizer("বাংলা ভাষায় একটি উদাহরণ"))

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Tokenizer loaded: google-bert/bert-base-multilingual-cased
{'input_ids': [101, 49133, 28590, 14374, 941, 106352, 26145, 40652, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [12]:
label_names = [
    "Positive",
    "Happiness",
    "Negative",
    "Sadness",
    "Question"
]
id2label = {
    i: name for i, name in enumerate(label_names)
}

label2id = {
    name: i for i, name in enumerate(label_names)
}

In [13]:
MAX_LENGTH = 128
TEXT_COLUMN = "sentence"

def tokenize_function(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation = True,
        max_length = MAX_LENGTH
    )

tokenized_dataset = hf_dataset.map(
    tokenize_function,
    batched = True,
     remove_columns=[TEXT_COLUMN],
)

print(tokenized_dataset)
print(tokenized_dataset["train"][0])

Map:   0%|          | 0/4145 [00:00<?, ? examples/s]

Map:   0%|          | 0/518 [00:00<?, ? examples/s]

Map:   0%|          | 0/519 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4145
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 518
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 519
    })
})
{'labels': [1.0, 1.0, 0.0, 0.0, 0.0], 'input_ids': [101, 52038, 14998, 968, 81837, 16431, 16755, 953, 12235, 111232, 12235, 36715, 968, 102712, 972, 11737, 36715, 949, 16166, 97303, 15613, 971, 30806, 941, 111232, 13458, 920, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels = len(label_names),
    problem_type = "multi_label_classification",
    id2label = id2label,
    label2id=label2id
)
print("Model loaded:", BASE_MODEL)
print("Number of classes:", model.config.num_labels)
print("Model label mapping:", model.config.id2label)

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

Model loaded: google-bert/bert-base-multilingual-cased
Number of classes: 5
Model label mapping: {0: 'Positive', 1: 'Happiness', 2: 'Negative', 3: 'Sadness', 4: 'Question'}


In [15]:

import numpy as np
from scipy.special import expit
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score
)

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred

    # Convert logits to probabilities using sigmoid
    probabilities = expit(logits)

    # Convert probabilities into binary predictions
    predictions = (probabilities >= 0.5).astype(int)

    # Ensure labels are in the expected binary format
    true_labels = np.asarray(true_labels).astype(int)

    # Macro metrics: calculate each label's score,
    # then give all five labels equal weight
    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    # Micro F1: calculate overall performance across all labels
    f1_micro = f1_score(
        true_labels,
        predictions,
        average="micro",
        zero_division=0
    )

    # Exact match: all five labels must be correct
    exact_match_accuracy = accuracy_score(
        true_labels,
        predictions
    )

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "exact_match_accuracy": exact_match_accuracy
    }

In [16]:
import transformers
import inspect

print("Transformers version:", transformers.__version__)
print(inspect.signature(TrainingArguments.__init__))

Transformers version: 5.22.2
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_co

In [17]:
import math

# Local training output
OUTPUT_DIR = "./bengali-sentiment-output"

# Training configuration
BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2
EPOCHS = 5

# Calculate total training steps
steps_per_epoch = math.ceil(
    math.ceil(len(tokenized_dataset["train"]) / BATCH_SIZE)
    / GRADIENT_ACCUMULATION
)

total_training_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_training_steps * 0.1)

print("Steps per epoch:", steps_per_epoch)
print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,

    # Evaluation and checkpoints
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    save_total_limit=2,

    # Hugging Face Hub upload
    push_to_hub=True,
    hub_model_id=NEW_MODEL_REPO,

    # Reproducibility and logging
    seed=42,
    data_seed=42,
    logging_steps=20,
    report_to="none",

    # Precision
    fp16=False,
)

Steps per epoch: 260
Total training steps: 1300
Warmup steps: 130


In [18]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [19]:
train_result = trainer.train()

print("\nTraining completed.")
print(train_result.metrics)

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Precision Macro,Recall Macro,Exact Match Accuracy
1,0.424568,0.228281,0.792917,0.835821,0.829686,0.785589,0.683398
2,0.335677,0.182327,0.852898,0.879674,0.888596,0.831468,0.756757
3,0.257981,0.146692,0.897281,0.911427,0.927901,0.870926,0.810811
4,0.202380,0.143837,0.891089,0.909831,0.929760,0.867690,0.806950
5,0.139358,0.137612,0.905538,0.918737,0.929521,0.885037,0.816602


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte


Training completed.
{'train_runtime': 311.7078, 'train_samples_per_second': 66.489, 'train_steps_per_second': 4.171, 'total_flos': 411090235637436.0, 'train_loss': 0.34004899153342616, 'epoch': 5.0}


In [20]:
import os
import glob
from safetensors.torch import load_file

best_checkpoint = trainer.state.best_model_checkpoint

print("Best checkpoint:", best_checkpoint)

checkpoint_files = glob.glob(
    os.path.join(best_checkpoint, "*.safetensors")
)

for file in checkpoint_files:
    state_dict = load_file(file, device="cpu")

    layernorm_keys = [
        key for key in state_dict.keys()
        if "embeddings.LayerNorm" in key
    ]

    print("\nFile:", file)
    print("LayerNorm keys:", layernorm_keys)

Best checkpoint: ./bengali-sentiment-output/checkpoint-1300

File: ./bengali-sentiment-output/checkpoint-1300/model.safetensors
LayerNorm keys: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma']


In [21]:
validation_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["validation"]
)

print("Validation results:")
for metric, value in validation_results.items():
    print(f"{metric}: {value}")

Training Loss,Validation Loss,Epoch,F1 Macro,F1 Micro,Precision Macro,Recall Macro,Exact Match Accuracy
0.139358,0.137612,5,0.905538,0.918737,0.929521,0.885037,0.816602


Validation results:
eval_loss: 0.1376122683286667
eval_f1_macro: 0.9055384628796672
eval_f1_micro: 0.9187374076561451
eval_precision_macro: 0.929521083948341
eval_recall_macro: 0.8850371310272516
eval_exact_match_accuracy: 0.8166023166023166


In [22]:
test_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

print("\nFinal test results:")
for metric, value in test_results.items():
    print(f"{metric}: {value}")

Training Loss,Validation Loss,Epoch,F1 Macro,F1 Micro,Precision Macro,Recall Macro,Exact Match Accuracy
0.139358,0.127689,5,0.916908,0.929596,0.932701,0.902837,0.828516



Final test results:
test_loss: 0.1276894062757492
test_f1_macro: 0.9169080957105205
test_f1_micro: 0.9295958279009127
test_precision_macro: 0.9327009891453185
test_recall_macro: 0.9028365191207028
test_exact_match_accuracy: 0.8285163776493256


In [23]:
import numpy as np
from scipy.special import expit
from sklearn.metrics import classification_report

predictions = trainer.predict(tokenized_dataset["test"])

# Convert logits to probabilities
probabilities = expit(predictions.predictions)

# Apply the baseline threshold
y_pred = (probabilities >= 0.5).astype(int)

# Actual labels
y_true = predictions.label_ids.astype(int)

print(classification_report(
    y_true,
    y_pred,
    target_names=label_names,
    digits=4,
    zero_division=0
))

              precision    recall  f1-score   support

    Positive     0.9948    0.9797    0.9872       197
   Happiness     0.8760    0.8760    0.8760       121
    Negative     0.9843    1.0000    0.9921       188
     Sadness     0.8083    0.7029    0.7519       138
    Question     1.0000    0.9556    0.9773       135

   micro avg     0.9444    0.9153    0.9296       779
   macro avg     0.9327    0.9028    0.9169       779
weighted avg     0.9417    0.9153    0.9277       779
 samples avg     0.9566    0.9358    0.9347       779



In [24]:
import numpy as np
from scipy.special import expit
from sklearn.metrics import f1_score

# Get validation predictions
val_predictions = trainer.predict(tokenized_dataset["validation"])

val_probabilities = expit(val_predictions.predictions)
y_val = val_predictions.label_ids.astype(int)

# Search for the best threshold independently for each label
thresholds = np.arange(0.10, 0.91, 0.01)

best_thresholds = {}

for i, label in enumerate(label_names):

    best_f1 = 0
    best_threshold = 0.5

    for threshold in thresholds:

        y_pred_label = (
            val_probabilities[:, i] >= threshold
        ).astype(int)

        score = f1_score(
            y_val[:, i],
            y_pred_label,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    best_thresholds[label] = best_threshold

    print(
        f"{label}: Threshold = {best_threshold:.2f}, "
        f"Validation F1 = {best_f1:.4f}"
    )

Positive: Threshold = 0.15, Validation F1 = 0.9792
Happiness: Threshold = 0.47, Validation F1 = 0.8841
Negative: Threshold = 0.39, Validation F1 = 0.9824
Sadness: Threshold = 0.37, Validation F1 = 0.7336
Question: Threshold = 0.12, Validation F1 = 0.9797


In [25]:
import numpy as np
from sklearn.metrics import classification_report

# Convert threshold dictionary into an ordered array
threshold_array = np.array([
    best_thresholds[label] for label in label_names
])

# Apply individual thresholds to test probabilities
y_pred_tuned = (
    probabilities >= threshold_array
).astype(int)

print("Test results with optimized thresholds:\n")

print(classification_report(
    y_true,
    y_pred_tuned,
    target_names=label_names,
    digits=4,
    zero_division=0
))

Test results with optimized thresholds:

              precision    recall  f1-score   support

    Positive     0.9949    0.9898    0.9924       197
   Happiness     0.8760    0.8760    0.8760       121
    Negative     0.9843    1.0000    0.9921       188
     Sadness     0.7537    0.7319    0.7426       138
    Question     0.9701    0.9630    0.9665       135

   micro avg     0.9278    0.9243    0.9260       779
   macro avg     0.9158    0.9121    0.9139       779
weighted avg     0.9269    0.9243    0.9255       779
 samples avg     0.9454    0.9422    0.9310       779



In [27]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model and tokenizer saved to:", OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to: ./bengali-sentiment-output


In [28]:
trainer.push_to_hub(
    commit_message="Upload fine-tuned Bengali sentiment model"
)

tokenizer.push_to_hub(NEW_MODEL_REPO)

print("Upload requested for:", NEW_MODEL_REPO)
print(f"https://huggingface.co/{NEW_MODEL_REPO}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


README.md:   0%|          | 0.00/2.44k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Upload requested for: Subha95/bengali-sentiment-bert
https://huggingface.co/Subha95/bengali-sentiment-bert


In [30]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=NEW_MODEL_REPO,
    tokenizer=NEW_MODEL_REPO,
    device=0 if torch.cuda.is_available() else -1,
)

sample_texts = [
    "আজ আমার খুব ভালো লাগছে",
    "আমি খুব দুঃখিত",
    "এটি মোটেও ভালো নয়",
    "তুমি কি আজ আসবে",
]

predictions = classifier(
    sample_texts,
    truncation=True,
    max_length=MAX_LENGTH,
)

for text, prediction in zip(sample_texts, predictions):
    print(f"Text: {text}")
    print(f"Predicted label: {prediction['label']}")
    print(f"Confidence: {prediction['score']:.4f}")
    print("-" * 50)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: আজ আমার খুব ভালো লাগছে
Predicted label: Happiness
Confidence: 0.9827
--------------------------------------------------
Text: আমি খুব দুঃখিত
Predicted label: Sadness
Confidence: 0.9810
--------------------------------------------------
Text: এটি মোটেও ভালো নয়
Predicted label: Negative
Confidence: 0.9930
--------------------------------------------------
Text: তুমি কি আজ আসবে
Predicted label: Question
Confidence: 0.8580
--------------------------------------------------
